In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import psycopg2
import schedule
import time
from datetime import datetime, timedelta
from decimal import Decimal

EVOLUTION_API_URL = "http://localhost:8080"

In [2]:
load_dotenv('.env.prod')

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS")
)

In [3]:
query_peca_hora="""
SELECT *
	FROM analytics.pecas_coletas_por_hora
"""

In [4]:
df = pd.read_sql_query(query_peca_hora, conn)

C:\Users\Luiz Felipe\AppData\Local\Temp\ipykernel_22680\326258466.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query_peca_hora, conn)


In [5]:
def formatar_mensagem_peca_hora(df):
    """Formata dados de peças por hora em mensagem WhatsApp formatada."""
    hora_atual = datetime.now().hour

    df = df.copy()
    df['DataHora_dt'] = pd.to_datetime(df['DataHora'])
    df['hora'] = df['DataHora_dt'].dt.hour
    df['qtde'] = df['qtde'].astype(int)

    df = df[(df['hora'] >= 3) & (df['hora'] <= hora_atual)]

    if df.empty:
        return None

    linhas = ["ENTRADA PÇS H. A HORA\n"]

    for hora in sorted(df['hora'].unique()):
        df_hora = df[df['hora'] == hora]
        total_hora = int(df_hora['qtde'].sum())

        linhas.append(f"{hora}h | {total_hora} PÇS |")
        linhas.append("")

        for _, row in df_hora.sort_values('tipo_tratado').iterrows():
            linhas.append(f"{row['tipo_tratado']} |{int(row['qtde'])}")

        linhas.append("")

    total_geral = int(df['qtde'].sum())
    total_por_tipo = df.groupby('tipo_tratado')['qtde'].sum().sort_values(ascending=False)
    itens = list(total_por_tipo.items())

    linhas.append(f"*Total Geral: {total_geral} PÇS*")
    for i, (tipo, qtde) in enumerate(itens):
        suffix = "*" if i == len(itens) - 1 else ""
        linhas.append(f"{tipo} |{int(qtde)}{suffix}")

    return "\n".join(linhas)

In [6]:
def formatar_mensagem_entregas(conn):
    """Formata entregas de hoje e amanhã (chat_1) em mensagem para grupo_2."""

    def fmt(dados):
        if dados is None:
            return "-"
        if isinstance(dados, (int, float, Decimal)):
            return f"{float(dados):.2f}"
        return str(dados)

    def emoji_status(entrega):
        if entrega == "ABERTA":
            return "\u274c"
        elif entrega == "FECHADA":
            return "\u2705"
        return "-"

    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            COALESCE(LEFT(c.alias, 20), 'Sem Cliente') AS "Cliente",
            ROUND(cee.realweight::numeric, 3) AS "Peso Sujo",
            ROUND(cex.realweight::numeric, 3) AS "Peso Limpo",
            CASE
                WHEN cex.closedate IS NULL THEN 'ABERTA'
                ELSE 'FECHADA'
            END AS status_entrega
        FROM clotheenterbarcode cee
        JOIN truckcollect tc ON tc.id = cee.truckcollect_id
        LEFT JOIN clotheexitbarcode cex ON cex.clotheenterbarcode_id = cee.id
        JOIN client c ON c.id = cee.client_id
        WHERE
            cex.id IS NOT NULL
            AND cex.enterdate::date = CURRENT_DATE
            AND cex.closedate IS NULL
        ORDER BY status_entrega ASC;
    """)
    registros_hoje = cursor.fetchall()

    cursor.execute("""
        SELECT
            COALESCE(LEFT(c.alias, 20), 'Sem Cliente') AS "Cliente",
            ROUND(cee.realweight::numeric, 3) AS "Peso Sujo",
            ROUND(cex.realweight::numeric, 3) AS "Peso Limpo",
            CASE
                WHEN cex.closedate IS NULL THEN 'ABERTA'
                ELSE 'FECHADA'
            END AS status_entrega
        FROM clotheenterbarcode cee
        JOIN truckcollect tc ON tc.id = cee.truckcollect_id
        LEFT JOIN clotheexitbarcode cex ON cex.clotheenterbarcode_id = cee.id
        JOIN client c ON c.id = cee.client_id
        WHERE
            cex.id IS NOT NULL
            AND cex.enterdate::date = (CURRENT_DATE + INTERVAL '1 day')
            AND cex.closedate IS NULL
        ORDER BY status_entrega ASC;
    """)
    registros_amanha = cursor.fetchall()
    cursor.close()

    hoje = datetime.now()
    amanha = hoje + timedelta(days=1)

    linhas = []
    linhas.append(f"DIA {hoje.strftime('%d')}")
    linhas.append("")
    linhas.append("CLIENTE | SUJO | LIMPO | STATUS")
    linhas.append("")
    for reg in registros_hoje:
        linhas.append(f"{fmt(reg[0])} | {fmt(reg[1])} | {fmt(reg[2])} | {emoji_status(reg[3])}")

    linhas.append("")
    linhas.append("-" * 30)
    linhas.append("")

    linhas.append(f"DIA {amanha.strftime('%d')}")
    linhas.append("")
    linhas.append("CLIENTE | SUJO | LIMPO | STATUS")
    linhas.append("")
    for reg in registros_amanha:
        linhas.append(f"{fmt(reg[0])} | {fmt(reg[1])} | {fmt(reg[2])} | {emoji_status(reg[3])}")

    return "\n".join(linhas)

In [7]:
def formatar_mensagem_relave(conn):
    """Formata relave da planta hora a hora (chat_2) em mensagem para grupo_3."""

    def fmt1(dados):
        if dados is None:
            return "0"
        if isinstance(dados, (int, float, Decimal)):
            return f"{float(dados):.1f}"
        return str(dados)

    def fmt2(dados):
        if dados is None:
            return "0.00"
        if isinstance(dados, (int, float, Decimal)):
            return f"{float(dados):.2f}"
        return str(dados)

    agora = datetime.now()
    hora_atual = agora.hour
    minuto_atual = agora.minute
    hora_inicial = 6

    cursor = conn.cursor()

    query_base = """
        WITH dados AS (
            SELECT
                TO_TIMESTAMP(\"DataHora\", 'DD/MM/YYYY HH24:MI') AS ts,
                \"Peso\",
                \"Tipo de roupa\"
            FROM analytics.relave_peso_ult6meses
            WHERE TRIM(UPPER(\"Cliente\")) = 'PLANTA LAVARE'
              AND TRIM(UPPER(\"Tipo de relave\")) = 'RELAVE-PLANTA'
        )
        SELECT SUM(
            CASE
                WHEN TRIM(UPPER(\"Tipo de roupa\")) = 'LEN\u00c7OL MOLHADO' THEN \"Peso\" * 0.60
                ELSE \"Peso\"
            END
        ) AS soma_ajustada
        FROM dados
        WHERE ts >= %s AND ts < %s;
    """

    linhas = ["RELAVE PLANTA H. A HORA"]
    subTotal = 0.0

    for h in range(hora_inicial, hora_atual):
        inicio = agora.replace(hour=h, minute=0, second=0, microsecond=0)
        fim = inicio + timedelta(hours=1)
        cursor.execute(query_base, (inicio, fim))
        soma = cursor.fetchone()[0]
        if soma is not None:
            subTotal += float(soma)
        linhas.append(f"{h}h | {fmt1(soma)} Kg")

    if hora_atual == 23 and 50 <= minuto_atual <= 59:
        inicio_23 = agora.replace(hour=23, minute=0, second=0, microsecond=0)
        fim_23 = agora.replace(second=0, microsecond=0) + timedelta(minutes=1)
        cursor.execute(query_base, (inicio_23, fim_23))
        soma_23 = cursor.fetchone()[0]
        if soma_23 is not None:
            subTotal += float(soma_23)
        linhas.append(f"23h (at\u00e9 {minuto_atual:02d}m) | {fmt1(soma_23)} Kg")

    cursor.execute("""
        SELECT COALESCE(SUM(\"Peso Coleta\"), 0) AS peso_coleta_hoje
        FROM analytics.peso_diario_coleta_entrega_por_carrinho
        WHERE TO_DATE(TRIM(\"Data\"), 'DD/MM/YYYY') = CURRENT_DATE;
    """)
    coleta_row = cursor.fetchone()
    coleta_val = float(coleta_row[0]) if coleta_row and coleta_row[0] is not None else 0.0
    cursor.close()

    porcento = (subTotal / coleta_val * 100) if coleta_val > 0 else 0.0

    linhas.append("")
    linhas.append(f"Total: {fmt1(subTotal)} Kg")
    linhas.append(f"Total Coletado: {fmt1(coleta_val)} Kg")
    linhas.append(f"% Relave: {fmt2(porcento)} %")

    return "\n".join(linhas)

In [8]:
def enviar_mensagem_whatsapp(mensagem, group_env_var="grupo_1"):
    instance = os.getenv("instancia_nome")
    api_key = os.getenv("API_KEY")
    group_id = os.getenv(group_env_var)

    url = f"{EVOLUTION_API_URL}/message/sendText/{instance}"
    headers = {"Content-Type": "application/json", "apikey": api_key}
    payload = {"number": group_id, "text": mensagem}

    response = requests.post(url, json=payload, headers=headers, timeout=30)
    print(url)
    print(payload)
    return response

In [ ]:


def job():
    timestamp = datetime.now().strftime('%d/%m %H:%M')

    # grupo_1: peças por hora
    df = pd.read_sql_query(query_peca_hora, conn)
    msg1 = formatar_mensagem_peca_hora(df)
    if msg1:
        r1 = enviar_mensagem_whatsapp(msg1, "grupo_1")
        print(f"[{timestamp}] grupo_1 — Status: {r1.status_code}")
    else:
        print(f"[{timestamp}] grupo_1 — Sem dados.")

    # grupo_2: entregas hoje e amanhã
    msg2 = formatar_mensagem_entregas(conn)
    if msg2:
        r2 = enviar_mensagem_whatsapp(msg2, "grupo_2")
        print(f"[{timestamp}] grupo_2 — Status: {r2.status_code}")
    else:
        print(f"[{timestamp}] grupo_2 — Sem dados.")

    # grupo_3: relave por hora
    msg3 = formatar_mensagem_relave(conn)
    if msg3:
        r3 = enviar_mensagem_whatsapp(msg3, "grupo_3")
        print(f"[{timestamp}] grupo_3 — Status: {r3.status_code}")
    else:
        print(f"[{timestamp}] grupo_3 — Sem dados.")


# Agenda de 03h às 22h, todo dia no início de cada hora
for hora in range(3, 23):
    schedule.every().day.at(f"{hora:02d}:00").do(job)

print("Agendamento ativo — 03h às 22h, a cada hora.")
print("Mantenha essa célula rodando. Ctrl+C para parar.\n")

job()  # Dispara imediatamente na primeira execução

while True:
    schedule.run_pending()
    time.sleep(30)

Agendamento ativo — 03h às 22h, a cada hora.
Mantenha essa célula rodando. Ctrl+C para parar.



C:\Users\Luiz Felipe\AppData\Local\Temp\ipykernel_22680\2532121621.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query_peca_hora, conn)


[22/06 00:01] grupo_1 — Sem dados.
http://192.168.250.146:8080/message/sendText/BotMensagens
{'number': '120363422945556874@g.us', 'text': 'DIA 22\n\nCLIENTE | SUJO | LIMPO | STATUS\n\nHEMO PAÇO | - | - | ❌\nH.T.O SLZ | 109.90 | - | ❌\nS.V.O SLZ | 22.40 | - | ❌\nH.C.M | 1118.20 | 409.51 | ❌\nH.CRIANÇA COLINAS | 86.80 | - | ❌\nRAIMUNDO LIMA | 66.30 | - | ❌\nMAT MAIOBÃO | 129.20 | - | ❌\nJUVÊNCIO MATOS | 399.40 | - | ❌\nBENEDITO LEITE | 318.70 | - | ❌\nPOLIC. DUTRA | - | - | ❌\nBALSAS | 449.20 | 226.50 | ❌\nUPA PARQUE VITORIA | 69.10 | - | ❌\nUPA PAÇO | 66.90 | - | ❌\nH. DA ILHA | 596.10 | - | ❌\nH.C.I  | 114.10 | - | ❌\nUPA ITAQUI BACANGA | 45.90 | - | ❌\nNINA RODRIGUES | 174.90 | - | ❌\nHOSP. DA CRIANÇA SLZ | 317.80 | - | ❌\nHCID SLZ | 877.00 | - | ❌\nHCID - ANEXO | 149.50 | - | ❌\nPOLIC. COHATRAC | 40.30 | - | ❌\nBARRA DO CORDA | 527.80 | 256.80 | ❌\nPRESIDENTE DUTRA  | 686.70 | 147.20 | ❌\nUPA VINHAIS | 66.40 | - | ❌\nDIAMANTE | - | - | ❌\nPRISIONAL | 41.00 | - | ❌\nAQUILES LISBOA | 